In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("kc_house_data.csv")
print(df.shape)
df.info()

# Date se features banao
df["date"] = pd.to_datetime(df["date"])
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df["house_age"] = df["sale_year"] - df["yr_built"]
df["was_renovated"] = (df["yr_renovated"] > 0).astype(int)

# Outlier hatao (33 bedrooms wala ghar galat entry hai)
df = df[df["bedrooms"] < 15]

# Useless columns hatao
df = df.drop(columns=["id", "date"])

In [ ]:
# Price distribution (right-skewed hai)
sns.histplot(df["price"], bins=50, kde=True)
plt.show()

# Kaunse features price se related hain
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=False, cmap="coolwarm")
plt.show()
print(df.corr()["price"].sort_values(ascending=False))

# Location map: mehenge ghar kahan hain
plt.scatter(df["long"], df["lat"], c=df["price"], cmap="viridis", s=2)
plt.colorbar(label="Price")
plt.show()

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["price"])
y = np.log1p(df["price"])     # log transform

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "R2": r2_score(y_test, pred),
        "MAE ($)": mean_absolute_error(np.expm1(y_test), np.expm1(pred)),
    })

print(pd.DataFrame(results).sort_values("R2", ascending=False))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    "n_estimators": [300, 500],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 4, 5],
    "subsample": [0.8, 1.0],
}

search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    params, n_iter=8, cv=3, scoring="r2", n_jobs=-1, random_state=42
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:", search.best_params_)
print("Test R2:", r2_score(y_test, best_model.predict(X_test)))

In [ ]:
imp = pd.Series(best_model.feature_importances_, index=X.columns)
imp.sort_values().plot(kind="barh", figsize=(8, 6))
plt.title("Feature Importance")
plt.show()

In [ ]:
import joblib
joblib.dump(best_model, "model.pkl")
joblib.dump(list(X.columns), "columns.pkl")   # column order yaad rakhne ke liye